In [1]:
!python3 -m pip install joblib


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip


In [2]:
!pip3 install scikit-learn


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip


In [3]:
import cv2
import os
import time

# Enter student name
student_name = input("Enter Student Name: ")

# Create folder
save_path = os.path.join("image", student_name)
os.makedirs(save_path, exist_ok=True)

# Load face detector
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    "haarcascade_frontalface_default.xml"
)

# Start webcam
cap = cv2.VideoCapture(0)

count = 0

print("Capturing 20 face images...")
print("Look at the camera and slightly change your pose.")

while True:

    ret, frame = cap.read()

    if not ret:
        print("Camera Error!")
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5,
        minSize=(50, 50)
    )

    for (x, y, w, h) in faces:

        # Capture original color face
        face = frame[y:y+h, x:x+w]

        # Resize face
        face = cv2.resize(face, (100, 100))

        count += 1

        file_name = os.path.join(
            save_path,
            f"{count}.jpg"
        )

        cv2.imwrite(file_name, face)

        # Draw rectangle
        cv2.rectangle(
            frame,
            (x, y),
            (x+w, y+h),
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"Captured: {count}/20",
            (10, 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2
        )

        cv2.imshow("Face Capture", frame)

        # Delay so photos are not identical
        time.sleep(0.5)

        if count >= 20:
            break

    cv2.imshow("Face Capture", frame)

    if count >= 20:
        break

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

print(f"\n20 images saved successfully for {student_name}")
print(f"Location: {save_path}")

Capturing 20 face images...
Look at the camera and slightly change your pose.

20 images saved successfully for himanshu
Location: image/himanshu


In [4]:
import os
import cv2
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib

DATASET_PATH = "image"

X = []
y = []

for person in os.listdir(DATASET_PATH):

    person_folder = os.path.join(DATASET_PATH, person)

    if not os.path.isdir(person_folder):
        continue

    for img_name in os.listdir(person_folder):

        img_path = os.path.join(person_folder, img_name)

        img = cv2.imread(img_path)

        if img is None:
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # Resize all images to same size
        gray = cv2.resize(gray, (100, 100))

        # Convert image to 1D vector
        features = gray.flatten()

        X.append(features)
        y.append(person)

X = np.array(X)
y = np.array(y)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train KNN model
model = KNeighborsClassifier(n_neighbors=3)

model.fit(X_train, y_train)

# Check accuracy
pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))

# Save model
joblib.dump(model, "face_model.pkl")

print("Model Saved Successfully!")

Accuracy: 1.0
Model Saved Successfully!


In [5]:
import cv2
import numpy as np
import pandas as pd
import joblib
import os
from datetime import datetime

# Load trained model
model = joblib.load("face_model.pkl")

# Face detector
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    "haarcascade_frontalface_default.xml"
)

# Dataset folder
DATASET_PATH = "image"   # change if your folder name is different

# Get all student names automatically
all_students = [
    folder for folder in os.listdir(DATASET_PATH)
    if os.path.isdir(os.path.join(DATASET_PATH, folder))
]

attendance_file = "attendance.csv"

present_students = set()

cap = cv2.VideoCapture(0)

print("Press 'q' to stop attendance")

while True:

    ret, frame = cap.read()

    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.3,
        minNeighbors=5,
        minSize=(50, 50)
    )

    for (x, y, w, h) in faces:

        # IMPORTANT: grayscale face
        face = gray[y:y+h, x:x+w]

        face = cv2.resize(face, (100, 100))

        sample = face.flatten().reshape(1, -1)

        try:

            prediction = model.predict(sample)[0]

            distances, _ = model.kneighbors(
                sample,
                n_neighbors=1
            )

            distance = distances[0][0]

            print(
                f"Prediction: {prediction} | Distance: {distance:.2f}"
            )

            # Unknown face threshold
            THRESHOLD = 4500

            if distance > THRESHOLD:

                name = "Unknown"
                color = (0, 0, 255)  # Red

            else:

                name = prediction
                color = (0, 255, 0)  # Green

                present_students.add(name)

        except Exception as e:

            print("Error:", e)

            name = "Unknown"
            color = (0, 0, 255)

        # Draw rectangle
        cv2.rectangle(
            frame,
            (x, y),
            (x + w, y + h),
            color,
            2
        )

        cv2.putText(
            frame,
            name,
            (x, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            color,
            2
        )

    cv2.imshow(
        "Attendance System",
        frame
    )

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

# Create attendance records
current_time = datetime.now().strftime("%H:%M:%S")

records = []

for student in all_students:

    if student in present_students:
        status = "Present"
    else:
        status = "Absent"

    records.append(
        [student, current_time, status]
    )

df = pd.DataFrame(
    records,
    columns=[
        "Name",
        "Time",
        "Status"
    ]
)

df.to_csv(
    attendance_file,
    index=False
)

print("\nAttendance Saved Successfully!\n")
print(df)

Press 'q' to stop attendance
Prediction: himanshu | Distance: 4111.77
Prediction: himanshu | Distance: 3295.01
Prediction: himanshu | Distance: 3607.96
Prediction: himanshu | Distance: 3399.45
Prediction: himanshu | Distance: 3183.84
Prediction: himanshu | Distance: 3105.15
Prediction: himanshu | Distance: 2856.92
Prediction: himanshu | Distance: 2299.37
Prediction: himanshu | Distance: 1449.42
Prediction: himanshu | Distance: 1953.23
Prediction: himanshu | Distance: 2487.89
Prediction: himanshu | Distance: 2552.64
Prediction: himanshu | Distance: 2403.89
Prediction: himanshu | Distance: 2448.70
Prediction: himanshu | Distance: 2494.16
Prediction: himanshu | Distance: 2566.65
Prediction: himanshu | Distance: 2569.86
Prediction: himanshu | Distance: 2735.44
Prediction: himanshu | Distance: 2408.55
Prediction: himanshu | Distance: 2471.32
Prediction: himanshu | Distance: 3029.32
Prediction: himanshu | Distance: 3020.06
Prediction: himanshu | Distance: 2351.25
Prediction: himanshu | Dista

In [6]:
!pip3 install --upgrade mysql-connector-python


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
